# HuggingFace Model Migration

This notebook shows how to migrate your Lightning checkpoint to a HuggingFace-compatible model format that decouples code from checkpoints.

## Why Migrate?

1. **Code-Checkpoint Decoupling**: Load models without maintaining source code
2. **Easy Sharing**: Push to HuggingFace Hub with one command
3. **Standard API**: Use `AutoModel.from_pretrained()` just like any other HF model
4. **Future-Proof**: Update code without affecting model loading

## Workflow Overview

1. Setup & Load Checkpoint
2. Extract Model Parameters
3. Create HuggingFace Model Directory
4. Save Model Code & Weights
5. Test Loading Locally
6. Verify Outputs Match Original
7. (Optional) Push to HuggingFace Hub
8. (Optional) Load from Hub with Variants

## Step 1: Setup - Load Your Current Checkpoint

In [ ]:
import os
os.chdir("../")

import sys
import torch
import json
import shutil
from pathlib import Path

In [ ]:
# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Point to your checkpoint - modify these paths based on your setup
# SNGP Models:
ckpt_path = "checkpoints/acevedo_sngp_resnet18/model.ckpt"
model_type = "sngp"  # Set to "sngp" or "baseline"

# Baseline Models (uncomment to use instead):
# ckpt_path = "checkpoints/acevedo_baseline_resnet18/model.ckpt"
# model_type = "baseline"

# Load checkpoint
checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
print(f"✓ Loaded checkpoint from: {ckpt_path}")
print(f"  Checkpoint keys: {checkpoint.keys()}")
print(f"  Model type: {model_type}")

In [ ]:
# Inspect state dict structure
state_dict = checkpoint['state_dict']
print(f"Number of parameters: {len(state_dict)}")
print(f"\nFirst 5 keys:")
for i, key in enumerate(list(state_dict.keys())[:5]):
    print(f"  {i+1}. {key}")

## Step 2: Extract Model Parameters from Checkpoint

The Lightning checkpoint has keys like `net.feature_extractor.layer1...` because they're wrapped. We need to remove the `net.` prefix and skip parametrization buffers (for SNGP models).

In [ ]:
def extract_state_dict(checkpoint, device):
    """
    Extract and remap state dict from Lightning checkpoint.
    
    - Removes 'net.' prefix (Lightning module wrapper)
    - Skips parametrization buffers (PyTorch spectral norm internals)
    - Keeps weight_orig, weight_u, weight_v (spectral norm components for SNGP)
    """
    state_dict = checkpoint['state_dict']
    
    remapped_state_dict = {}
    skipped_keys = []
    
    for key, value in state_dict.items():
        # Remove 'net.' prefix (Lightning module wrapper)
        new_key = key.replace('net.', '', 1)
        
        # Skip parametrization buffers created by PyTorch's spectral_norm wrapper
        # These are internal and will cause warnings if loaded
        if 'parametrizations' in new_key:
            skipped_keys.append(new_key)
            continue
        
        # Keep all other parameters/buffers (weight_orig, weight_u, weight_v, etc.)
        remapped_state_dict[new_key] = value
    
    if skipped_keys:
        print(f"⚠ Skipped {len(skipped_keys)} parametrization buffer keys (internal PyTorch)")
        print(f"  Example: {skipped_keys[0] if skipped_keys else 'None'}")
    
    return remapped_state_dict

# Extract the clean state dict
clean_state_dict = extract_state_dict(checkpoint, device)
print(f"\n✓ Extracted state dict:")
print(f"  Total keys: {len(clean_state_dict)}")

# For SNGP: Verify spectral norm components
if model_type == "sngp":
    weight_orig = sum(1 for k in clean_state_dict.keys() if k.endswith('.weight_orig'))
    weight_u = sum(1 for k in clean_state_dict.keys() if k.endswith('.weight_u'))
    weight_v = sum(1 for k in clean_state_dict.keys() if k.endswith('.weight_v'))
    parametrizations = sum(1 for k in clean_state_dict.keys() if 'parametrizations' in k)
    
    print(f"\n✓ Spectral norm components (SNGP):")
    print(f"  weight_orig: {weight_orig}")
    print(f"  weight_u: {weight_u}")
    print(f"  weight_v: {weight_v}")
    print(f"  parametrizations: {parametrizations} (should be 0)")

## Step 3: Create HuggingFace Model Directory Structure

In [ ]:
# Set model configuration based on your model type and dataset

if model_type == "sngp":
    # Acevedo dataset SNGP
    model_config = {
        "arch": "resnet18",
        "num_classes": 8,
        "rff_dim": 1024,
        "length_scale": 1.0,
        "ridge_penalty": 1e-3,
        "cov_momentum": 0.999,
        "mean_field": True,
        "n_power_iterations_sn": 1,
        "pretrained": False,
    }
    architectures = "SNGPForImageClassification"
    model_type_str = "sngp_classifier"
    hf_model_file = "hf_sngp_model"
else:  # baseline
    # Baseline model
    model_config = {
        "arch": "resnet18",
        "num_classes": 8,
        "dropout_p": 0.5,
        "pretrained": False,
    }
    architectures = "BaselineClassifierForImageClassification"
    model_type_str = "baseline_classifier"
    hf_model_file = "hf_model"

print(f"Model config:")
for key, val in model_config.items():
    print(f"  {key}: {val}")

In [ ]:
# Create output directory
# Modify this path based on where you want to save the HF model
if model_type == "sngp":
    hf_model_dir = Path("./checkpoints/acevedo_sngp_resnet18/hf_checkpoint")
else:
    hf_model_dir = Path("./hf_checkpoint")

hf_model_dir.mkdir(parents=True, exist_ok=True)

print(f"✓ Created directory: {hf_model_dir}")
print(f"  Path: {hf_model_dir.absolute()}")

In [ ]:
# Create config.json - tells HuggingFace how to load the model
config_dict = {
    "architectures": [architectures],
    "model_type": model_type_str,
    "auto_map": {
        "AutoConfig": f"{hf_model_file}.SNGPConfig" if model_type == "sngp" else f"{hf_model_file}.BaselineClassifierConfig",
        "AutoModel": f"{hf_model_file}.SNGPForImageClassification" if model_type == "sngp" else f"{hf_model_file}.BaselineClassifierForImageClassification",
    },
    **model_config,
}

config_path = hf_model_dir / "config.json"
with open(config_path, "w") as f:
    json.dump(config_dict, f, indent=2)

print(f"✓ Saved config.json")
print(f"\nConfig contents:")
print(json.dumps(config_dict, indent=2))

## Step 4: Save Model Code & Weights

This is the key step for decoupling! We save:
1. **config.json** - Model configuration ✓ (already done above)
2. **pytorch_model.bin** - Model weights
3. **hf_model.py** - Model code (no external dependencies)

In [ ]:
# Save weights
weights_path = hf_model_dir / "pytorch_model.bin"
torch.save(clean_state_dict, weights_path)

file_size_mb = weights_path.stat().st_size / 1e6
print(f"✓ Saved pytorch_model.bin")
print(f"  Size: {file_size_mb:.1f} MB")
print(f"  Keys saved: {len(clean_state_dict)}")

In [ ]:
# Copy the HF model code - this enables loading without src/ directory
if model_type == "sngp":
    hf_model_src = Path("src/hf_sngp_model.py")
else:
    hf_model_src = Path("src/hf_model.py")

hf_model_dst = hf_model_dir / hf_model_src.name

# Clear cached modules before copying
modules_to_remove = [m for m in list(sys.modules.keys()) 
                      if 'hf_' in m or 'transformers_modules' in m]
for m in modules_to_remove:
    del sys.modules[m]

shutil.copy(hf_model_src, hf_model_dst)

print(f"✓ Copied model code")
print(f"  Source: {hf_model_src}")
print(f"  Dest: {hf_model_dst}")

## Step 5: Test Loading the Model Locally

In [ ]:
from transformers import AutoConfig, AutoModel

# Load and display config
config = AutoConfig.from_pretrained(str(hf_model_dir), trust_remote_code=True)
print(f"✓ Loaded config")
print(f"  Model type: {config.model_type}")
print(f"  Architecture: {config.arch}")
print(f"  Num classes: {config.num_classes}")

In [ ]:
# Load model using HuggingFace API (this works WITHOUT your src/ directory!)
model = AutoModel.from_pretrained(str(hf_model_dir), trust_remote_code=True)
model = model.to(device)
model.eval()

print(f"✓ Loaded model successfully!")
print(f"  Type: {type(model).__name__}")
print(f"  Device: {next(model.parameters()).device}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Test inference with dummy input
test_input = torch.randn(2, 3, 224, 224).to(device)

with torch.no_grad():
    output = model(test_input)

print(f"✓ Inference successful!")
print(f"  Input shape: {test_input.shape}")
if isinstance(output, dict):
    print(f"  Output keys: {output.keys()}")
    for key in output.keys():
        val = output[key]
        if isinstance(val, torch.Tensor):
            print(f"    {key}: {val.shape}")
else:
    print(f"  Output shape: {output.shape}")

## Step 6: Verify Model Outputs Match Original

Load the original model and compare outputs to ensure perfect migration.

In [ ]:
# Load original model for comparison
if model_type == "sngp":
    from src.models.sngp_gpt import SNGPClassifier
    original_model = SNGPClassifier(
        num_classes=8,
        arch="resnet18",
        pretrained=False,
        rff_dim=1024,
        length_scale=1.0,
        ridge_penalty=1e-3,
        cov_momentum=0.999,
        mean_field=True,
        n_power_iterations_sn=1
    )
else:
    from src.models.baseline.baseline_models import BaselineClassifier
    original_model = BaselineClassifier(
        arch="resnet18",
        num_classes=8,
        dropout_p=0.5,
        pretrained=False
    )

original_model = original_model.to(device)
original_model.eval()

# Load checkpoint weights
original_state_dict = checkpoint['state_dict']
remapped = {}
for k, v in original_state_dict.items():
    remapped[k.replace('net.', '', 1)] = v

original_model.load_state_dict(remapped)
print(f"✓ Loaded original model")
print(f"  Type: {type(original_model).__name__}")

In [ ]:
# Compare outputs
with torch.no_grad():
    # Original model output
    if model_type == "sngp":
        original_out = original_model(test_input)  # Returns tuple
        original_logits = original_out[0]  # mean_field_logits
    else:
        original_logits = original_model(test_input)
    
    # HF model output
    hf_out = model(test_input)
    if isinstance(hf_out, dict):
        hf_logits = hf_out['logits']
    else:
        hf_logits = hf_out

# Compare
max_diff = (original_logits - hf_logits).abs().max().item()
is_match = torch.allclose(original_logits, hf_logits, atol=1e-5)

print(f"✓ Output comparison:")
print(f"  Max difference: {max_diff:.2e}")
print(f"  Outputs match: {is_match}")

if is_match:
    print(f"\n  ✓✓✓ SUCCESS! Outputs are numerically identical!")
else:
    print(f"\n  ⚠ Outputs differ - may need to debug model structure")

## Step 7: (Optional) Push to HuggingFace Hub

In [ ]:
# Check HuggingFace Hub login status
try:
    from huggingface_hub import get_token
    
    token = get_token()
    if token is None:
        print("⚠ Not logged into HuggingFace Hub")
        print("  To push models, run: huggingface-cli login")
    else:
        print("✓ Logged into HuggingFace Hub")
        print("  Ready to push models!")
        
except ImportError:
    print("⚠ huggingface_hub not installed")
    print("  Install with: pip install huggingface_hub")

In [ ]:
# Example: Push to HuggingFace Hub (uncomment and modify to use)
"""
from huggingface_hub import upload_folder

repo_id = "your-username/your-model-name"  # Change this!
upload_folder(
    repo_id=repo_id,
    folder_path=str(hf_model_dir),
    repo_type="model",
)

print(f"✓ Model pushed to: https://huggingface.co/{repo_id}")
"""

## Step 8: Usage Examples

Now you can use your model in three ways!

In [ ]:
# Example 1: Load from Local Directory
"""
from transformers import AutoModel

model = AutoModel.from_pretrained("./hf_checkpoint", trust_remote_code=True)
model.eval()

# Inference
input_tensor = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    output = model(input_tensor)
"""

print("See code cell for local loading example")